# Phase E — Golden Answers & Caveats

**Owner:** Member 2 (Mohammed)

Member 4 uses these as the chatbot evaluation set:

- **golden_answers.json** — machine-readable ground truth, one entry per canonical question.
  The chatbot's answer must match these numbers (target ≥90% accuracy).
- **golden_answers.md** — human-readable narrative. Lift sentences for the chatbot system prompt.
- **caveats.md** — the disclaimers the chatbot must surface on every answer.

All seven canonical questions are answered programmatically from the locked knowledge layer.

## 1. Setup

In [1]:
import sys
import json
from datetime import datetime
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

from src.data_io import PROCESSED_DIR
from src import golden

REPORTS = REPO_ROOT / 'reports'

kpis = pd.read_csv(PROCESSED_DIR / 'neighbourhood_kpis.csv')
bcn_feat = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_listings_features.csv')
ldn_feat = pd.read_csv(PROCESSED_DIR / 'london' / 'london_listings_features.csv')
features = pd.concat([bcn_feat, ldn_feat], ignore_index=True)
bcn_monthly = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_monthly_metrics_clean.csv', parse_dates=['month_date'])
ldn_monthly = pd.read_csv(PROCESSED_DIR / 'london' / 'london_monthly_metrics_clean.csv', parse_dates=['month_date'])
bcn_monthly['city'] = 'barcelona'
ldn_monthly['city'] = 'london'
monthly = pd.concat([bcn_monthly, ldn_monthly], ignore_index=True)
print(f'kpis: {kpis.shape}, features: {features.shape}, monthly: {monthly.shape}')

kpis: (560, 25), features: (12237, 50), monthly: (395643, 18)


## 2. Compute all 7 answers

In [2]:
answers = golden.compute_all_answers(kpis, features, monthly)
for q, a in answers.items():
    print(f'{q}: {a["question"][:80]}')

Q1: Q1 — Which neighbourhoods have the highest concentration of STRs?
Q2: Q2 — Where is Airbnb most likely removing homes from the long-term residential m
Q3: Q3 — Which neighbourhoods are dominated by commercial/professional hosts?
Q4: Q4 — In which neighbourhoods does high STR density coincide with high prices?
Q5: Q5 — Which neighbourhoods are saturated, and which are emerging hotspots?
Q6: Q6 — How does STR pressure compare between Barcelona and London?
Q7: Q7 — If entire-home listings were capped at X nights/year, how many listings and


## 3. Spot-check each answer

In [3]:
print('Q1 — top STR density:')
for city, rows in answers['Q1']['answers'].items():
    print(f'  {city}: top 3 -> {[(r["geo_key"], r["str_density"]) for r in rows[:3]]}')

Q1 — top STR density:
  barcelona: top 3 -> [("la Dreta de l'Eixample", 295), ('el Raval', 203), ('Sant Pere, Santa Caterina i la Ribera', 163)]
  london: top 3 -> [('Whitechapel', 242), ('Westbourne Green', 210), ('Marylebone', 178)]


In [4]:
print('Q2 — top entire-home share (min density 30):')
for city, rows in answers['Q2']['answers'].items():
    print(f'  {city}: top 3 -> {[(r["geo_key"], r["entire_home_share"]) for r in rows[:3]]}')

Q2 — top entire-home share (min density 30):
  barcelona: top 3 -> [('les Corts', 0.875), ("el Camp d'en Grassot i Gràcia Nova", 0.784), ('la Sagrada Família', 0.736)]
  london: top 3 -> [('Brompton', 0.95), ('London Borough of Wandsworth', 0.944), ('Mayfair', 0.939)]


In [5]:
print('Q3 — top commercial host share:')
for city, rows in answers['Q3']['answers'].items():
    print(f'  {city}: top 3 -> {[(r["geo_key"], r["commercial_host_share"]) for r in rows[:3]]}')

Q3 — top commercial host share:
  barcelona: top 3 -> [("la Nova Esquerra de l'Eixample", 0.404), ('el Poble-sec', 0.364), ("l'Antiga Esquerra de l'Eixample", 0.345)]
  london: top 3 -> [('Bayswater', 0.387), ('Notting Hill', 0.37), ("Earl's Court", 0.357)]


In [6]:
print('Q4 — density-price intersection:')
for city, rows in answers['Q4']['answers'].items():
    print(f'  {city}: {len(rows)} subdivisions in top quartile for both')
    print(f'    correlation density vs price: {answers["Q4"]["correlations"][city]}')
    for r in rows[:5]:
        print(f'    - {r["geo_key"]}: density={r["str_density"]}, price={r["median_nightly_price"]}')

Q4 — density-price intersection:
  barcelona: 1 subdivisions in top quartile for both
    correlation density vs price: 0.298
    - la Dreta de l'Eixample: density=295, price=265.6
  london: 12 subdivisions in top quartile for both
    correlation density vs price: 0.258
    - Westbourne Green: density=210, price=250.2
    - Marylebone: density=178, price=273.4
    - Earl's Court: density=168, price=245.2
    - Paddington: density=152, price=301.5
    - Fulham: density=139, price=235.7


In [7]:
print('Q5 — saturated neighbourhoods:')
for city, rows in answers['Q5']['answers']['saturated'].items():
    print(f'  {city}: top 3 -> {[r["geo_key"] for r in rows[:3]]}')
print('\nQ5 — emerging neighbourhoods (growth %):')
for city, rows in answers['Q5']['answers']['emerging'].items():
    print(f'  {city}: top 3 -> {[(r["geo_key"], r["active_growth_pct"]) for r in rows[:3]]}')

Q5 — saturated neighbourhoods:
  barcelona: top 3 -> ["la Dreta de l'Eixample", 'el Poble-sec', 'la Sagrada Família']
  london: top 3 -> ['Notting Hill', "Earl's Court", 'Westbourne Green']

Q5 — emerging neighbourhoods (growth %):
  barcelona: top 3 -> [('Can Baró', 20.0), ('la Sagrera', 0.0), ('la Font de la Guatlla', 0.0)]
  london: top 3 -> [('Greenford', 13.3), ('Abbey Wood', 11.4), ('Gants Hill', 10.0)]


In [8]:
print('Q6 — citywide totals:')
for city, vals in answers['Q6']['answers']['citywide_totals'].items():
    print(f'  {city}: listings={vals["total_listings"]:,}, entire_homes={vals["entire_homes"]:,} ({vals["entire_home_share"]*100:.1f}%), breach90={vals["breach_90_total"]:,}')

Q6 — citywide totals:
  barcelona: listings=2,594, entire_homes=1,539 (59.3%), breach90=642
  london: listings=9,643, entire_homes=6,536 (67.8%), breach90=1,485


In [9]:
print('Q7 — city totals at each cap:')
for city, vals in answers['Q7']['answers']['city_totals'].items():
    print(f'  {city}: 90n={vals["cap_90_listings_impacted"]:,}, 60n={vals["cap_60_listings_impacted"]:,}, 30n={vals["cap_30_listings_impacted"]:,}, RESIDE={vals["reside_unregistered_total"]:,}')
print('\nQ7 — top impacted at 90-night cap:')
for city, caps in answers['Q7']['answers']['top_impacted_neighbourhoods'].items():
    print(f'  {city}: top 3 -> {[(r["geo_key"], r["listings_impacted"]) for r in caps["cap_90"][:3]]}')

Q7 — city totals at each cap:
  barcelona: 90n=611, 60n=689, 30n=751, RESIDE=421
  london: 90n=1,482, 60n=1,863, 30n=2,260, RESIDE=2,563

Q7 — top impacted at 90-night cap:
  barcelona: top 3 -> [("la Dreta de l'Eixample", 95), ('la Sagrada Família', 53), ('el Poble-sec', 46)]
  london: top 3 -> [('Whitechapel', 44), ('Paddington', 42), ('Westbourne Green', 41)]


## 4. Save machine-readable JSON

In [10]:
out_json = REPORTS / 'golden_answers.json'
out_json.write_text(json.dumps(answers, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved -> {out_json.relative_to(REPO_ROOT)}')
print(f'Size: {out_json.stat().st_size:,} bytes')

Saved -> reports/golden_answers.json
Size: 39,989 bytes


## 5. Render human-readable markdown

In [11]:
def render_top_table(rows, headers):
    if not rows:
        return '_No results._'
    head = '| ' + ' | '.join(headers) + ' |'
    sep = '|' + '|'.join(['---'] * len(headers)) + '|'
    body = []
    for r in rows:
        body.append('| ' + ' | '.join(str(r.get(h.replace(' ', '_').lower(), '')) for h in headers) + ' |')
    return '\n'.join([head, sep] + body)

today = datetime.utcnow().strftime('%Y-%m-%d')
lines = ['# Golden Answers — 7 Canonical Questions', '',
         f'_Generated on {today}._', '',
         'Member 4 evaluates the chatbot against these numbers. Target: ≥90% accuracy. Numbers come directly from `data/processed/neighbourhood_kpis.csv`.',
         '', '---', '']

for q in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']:
    a = answers[q]
    lines.append(f'## {q} — {a["question"].split("—")[-1].strip()}')
    lines.append('')
    lines.append(f'_Method: {a["method"]}_')
    lines.append('')
    if q in ('Q1', 'Q2', 'Q3'):
        for city in ('barcelona', 'london'):
            rows = a['answers'][city]
            lines.append(f'### {city.title()}')
            lines.append('')
            df = pd.DataFrame(rows)
            lines.append(df.to_markdown(index=False))
            lines.append('')
    elif q == 'Q4':
        for city in ('barcelona', 'london'):
            lines.append(f'### {city.title()}')
            lines.append('')
            lines.append(f'Correlation between density and price: **{a["correlations"][city]}**')
            lines.append('')
            rows = a['answers'][city]
            if rows:
                lines.append(pd.DataFrame(rows).to_markdown(index=False))
            else:
                lines.append('_No subdivisions in top quartile of both._')
            lines.append('')
    elif q == 'Q5':
        for kind in ('saturated', 'emerging'):
            lines.append(f'### {kind.title()} — across both cities')
            lines.append('')
            for city in ('barcelona', 'london'):
                rows = a['answers'][kind][city]
                lines.append(f'**{city.title()}**')
                lines.append('')
                if rows:
                    lines.append(pd.DataFrame(rows).to_markdown(index=False))
                else:
                    lines.append('_No qualifying neighbourhoods._')
                lines.append('')
    elif q == 'Q6':
        lines.append('### Citywide totals')
        lines.append('')
        ctotals = a['answers']['citywide_totals']
        rows = []
        for city in ('barcelona', 'london'):
            rows.append({**{'city': city}, **ctotals[city]})
        lines.append(pd.DataFrame(rows).to_markdown(index=False))
        lines.append('')
        lines.append('### Neighbourhood-level medians')
        lines.append('')
        medians = a['answers']['neighbourhood_medians']
        rows = []
        for city in ('barcelona', 'london'):
            rows.append({**{'city': city}, **medians[city]})
        lines.append(pd.DataFrame(rows).to_markdown(index=False))
        lines.append('')
    elif q == 'Q7':
        lines.append('### City totals at each cap')
        lines.append('')
        rows = []
        for city in ('barcelona', 'london'):
            rows.append({**{'city': city}, **a['answers']['city_totals'][city]})
        lines.append(pd.DataFrame(rows).to_markdown(index=False))
        lines.append('')
        for cap in (90, 60, 30):
            lines.append(f'### Top impacted neighbourhoods — {cap}-night cap')
            lines.append('')
            for city in ('barcelona', 'london'):
                rows = a['answers']['top_impacted_neighbourhoods'][city][f'cap_{cap}']
                lines.append(f'**{city.title()}**')
                lines.append('')
                lines.append(pd.DataFrame(rows).to_markdown(index=False))
                lines.append('')
        lines.append('### RESIDE simulation — Barcelona top 10')
        lines.append('')
        lines.append(pd.DataFrame(a['answers']['reside_top_neighbourhoods_barcelona']).to_markdown(index=False))
        lines.append('')
    lines.append('---')
    lines.append('')

out_md = REPORTS / 'golden_answers.md'
out_md.write_text('\n'.join(lines), encoding='utf-8')
print(f'Saved -> {out_md.relative_to(REPO_ROOT)}')
print(f'Size: {out_md.stat().st_size:,} bytes')

Saved -> reports/golden_answers.md
Size: 27,031 bytes


## 6. Write caveats.md

In [12]:
lines = ['# Chatbot Caveats', '',
         f'_Generated on {today}._', '',
         'Disclaimers Member 4 must surface in the chatbot. These belong in:',
         '',
         '- The system prompt (every answer respects these caveats)',
         '- Per-answer footers when a specific caveat applies (e.g., Q7 → RESIDE caveat)',
         '- The static "About" / "Limitations" page of the Streamlit app',
         '',
         '## Caveats',
         '']
for cav in golden.CAVEATS:
    lines.append(f'### {cav["title"]}')
    lines.append('')
    lines.append(f'**Applies to:** {cav["applies_to"]}')
    lines.append('')
    lines.append(cav['body'])
    lines.append('')
out_cav = REPORTS / 'caveats.md'
out_cav.write_text('\n'.join(lines), encoding='utf-8')
print(f'Saved -> {out_cav.relative_to(REPO_ROOT)}')
print(f'Size: {out_cav.stat().st_size:,} bytes')

# Also save caveats as JSON for the chatbot to load programmatically
out_cav_json = REPORTS / 'caveats.json'
out_cav_json.write_text(json.dumps(golden.CAVEATS, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved -> {out_cav_json.relative_to(REPO_ROOT)}')

Saved -> reports/caveats.md
Size: 2,743 bytes
Saved -> reports/caveats.json


## 7. Final handover

Member 2 work fully complete. Member 4 picks up:

- `reports/golden_answers.json` — load directly into the eval harness
- `reports/golden_answers.md` — narrative; system prompt copy-source
- `reports/caveats.json` / `.md` — disclaimers for the chatbot

Member 3 has everything from Phase D for clustering + risk score.